In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms.functional as VF
from PIL import Image
import time

device = 'cuda'
print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.8.0+cu128


In [2]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [3]:
train_dataset = datasets.MNIST(
    root="./DATA",
    train=True,
    download=True,
    transform=transform)

test_dataset = datasets.MNIST(
    root="./DATA",
    train=False,
    download=True,
    transform=transform)    

In [4]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 256)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)        
        return x

In [5]:
def test_model(model, dataloader, criterion):
    model.eval()
    running_loss_test = 0.0
    n_obs_test = 0
    for data, target in dataloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        running_loss_test += loss.item() * len(data)
        n_obs_test += len(data)
    epoch_loss_test = running_loss_test / n_obs_test
    return epoch_loss_test

In [6]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=False)

In [7]:
model = SimpleNN().to(device)
optimizer = optim.Adam(
    params=model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)
criterion = nn.CrossEntropyLoss()

exp_folder = f'runs/model_{int(time.time())}'
writer_train = SummaryWriter(f"{exp_folder}/train")
writer_valid = SummaryWriter(f"{exp_folder}/valid")
global_step = 0
for epoch in range(10):
    running_loss = 0.0
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        running_loss += loss.item()

        # Log Params Batch
        total_param_norm = 0.0
        total_param_count = 0.0
        total_grad_norm = 0.0
        for name, param in model.named_parameters():
            if param.requires_grad:
                if "bias" in name:
                    short_name = name[:-5]
                    group_name = "Bias"
                else:
                    short_name = name[:-7]
                    group_name = "Weight"
                
                writer_train.add_scalar(f"{group_name}/{short_name}_norm_scaled", param.data.norm(2) / (param.data.numel() ** 0.5), global_step) 
                writer_train.add_scalar(f"Gradient_{group_name}/{short_name}_norm", param.grad.data.norm(2), global_step) 
                writer_train.add_scalar(f"Update_ratio_{group_name}/{short_name}", param.grad.data.norm(2) / param.data.norm(2), global_step)

                total_param_norm += param.data.norm(2) ** 2
                total_param_count += param.data.numel()
                total_grad_norm += param.grad.data.norm(2) ** 2
        writer_train.add_scalar(f"Total_norm/Param_scaled", (total_param_norm / total_param_count) ** 0.5, global_step) 
        writer_train.add_scalar(f"Total_norm/Gradient", total_grad_norm ** 0.5, global_step)
        writer_train.add_scalar("Loss/batch", loss.item(), global_step)
        
        global_step += 1
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step() 

    # Log Params Epoch
    for name, param in model.named_parameters():
        if param.requires_grad:
            if "bias" in name:
                short_name = name[:-5]
                group_name = "Bias"
            else:
                short_name = name[:-7]
                group_name = "Weight"
        writer_train.add_histogram(f"{group_name}/{short_name}_epoch", param.data.cpu(), epoch)
        writer_train.add_histogram(f"Gradient_{group_name}/{short_name}_epoch", param.grad.data.cpu(), epoch)
    loss_train_epoch = running_loss / len(train_loader)
    writer_train.add_scalar("Loss_epoch", loss_train_epoch, epoch)

    loss_valid_epoch = test_model(model = model, dataloader = test_loader, criterion = criterion)
    writer_valid.add_scalar("Loss_epoch", loss_valid_epoch, epoch)
    
writer.close()

NameError: name 'writer' is not defined

In [20]:
0.999 ** 1170

0.3101853086390573

In [12]:
import torch
import math

def get_adam_effective_lr(optimizer, step_num):
    """
    Compute the effective per-parameter learning rate in Adam.

    Args:
        optimizer: torch.optim.Adam (or AdamW)
        step_num: current optimization step (used for bias correction)

    Returns:
        Dict[param_name -> tensor of effective LRs]
    """
    lr_dict = {}

    for i, group in enumerate(optimizer.param_groups):
        base_lr = group["lr"]
        beta1, beta2 = group["betas"]
        eps = group["eps"]

        for p in group["params"]:
            if p not in optimizer.state:
                continue

            state = optimizer.state[p]
            if "exp_avg_sq" not in state:
                continue

            v = state["exp_avg_sq"]
            # Bias correction for v
            v_hat = v / (1 - beta2 ** step_num)
            eff_lr = base_lr / (torch.sqrt(v_hat) + eps)

            lr_dict[p] = eff_lr

    return lr_dict


In [14]:
lr_dict = get_adam_effective_lr(optimizer, global_step)

for p, lr_tensor in lr_dict.items():
    print(p.shape, lr_tensor.mean().item(), lr_tensor.std().item())

torch.Size([1024, 784]) 2274.83642578125 8002.7587890625
torch.Size([1024]) 3.2127578258514404 13.126164436340332
torch.Size([256, 1024]) 137.4449462890625 1054.7147216796875
torch.Size([256]) 5.298520565032959 11.927927017211914
torch.Size([10, 256]) 23.76293182373047 143.5838623046875
torch.Size([10]) 0.2814764976501465 0.04730387404561043
